# Perbandingan Nilai K pada Bayesian Smoothing (ETA Model)

Notebook ini dibuat khusus untuk membandingkan performa model dengan parameter smoothing $K=2$ dan $K=3$, membandingkannya dengan baseline $K=1$.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from transjogja_CRISP_DM import (
    canonicalize_stops,
    normalize_trip_times,
    build_segments,
    split_train_val_segments,
    build_segment_lookup,
    predict_segments,
    mae,
    rmse
)

# Load data
stop_times_raw = pd.read_csv("raw/transjogja_stop_times_final_v6.csv")
stops = canonicalize_stops()
stop_times_norm = normalize_trip_times(stop_times_raw)

# Build segments
segments = build_segments(stop_times_norm, stops)
train, val = split_train_val_segments(segments)

print(f"Total segmen: {len(segments):,}")
print(f"Data latih: {len(train):,} baris")
print(f"Data validasi: {len(val):,} baris")


In [ ]:
# Evaluasi K=1.0 (Baseline), K=2.0, K=3.0
best_bin_size = 3
best_min_bin_n = 3

results = []
for k in [1.0, 2.0, 3.0]:
    base_lookup, bin_lookup = build_segment_lookup(train, bin_size=best_bin_size, K=k, min_bin_n=best_min_bin_n)
    pred = predict_segments(val, base_lookup, bin_lookup, bin_size=best_bin_size)
    usable = pred.dropna(subset=['pred_min', 'travel_time_min'])
    
    current_mae = mae(usable['travel_time_min'], usable['pred_min'])
    current_rmse = rmse(usable['travel_time_min'], usable['pred_min'])
    
    results.append({
        'K': k,
        'MAE (menit)': current_mae,
        'RMSE (menit)': current_rmse
    })

df_results = pd.DataFrame(results)
print("=== Perbandingan Nilai K pada Bayesian Smoothing ===")
print(df_results.to_string(index=False))
